# Reference 1 — Collections and Data Models

A notebook of **six fully worked reference answers**. Read the reasoning, run the code, test alternate inputs, and then try to solve the same task without looking.

**Level:** Intermediate · **Mode:** Worked reference solutions · **Prerequisite:** Complete related numbered lessons first.

**Instructions:** Write clear functions or short programs, not just expressions. State assumptions, test at least one ordinary case and one edge case, and avoid external network calls. Some exercises have multiple valid designs.

## 01. Normalize distinct names

Normalize input without mutating the source. Use a set comprehension for uniqueness and sorted for repeatable presentation.

**Acceptance criteria:** No duplicates or blank values; stable order.

**Hint:** This uses O(n) expected set construction, followed by O(k log k) sorting.

In [ ]:
def unique_names(names):
    return sorted({name.strip().casefold() for name in names if name.strip()})
assert unique_names([" Ada ", "ADA", "Bea", ""]) == ["ada", "bea"]
print(unique_names(["Cleo", "cleO", "Ada"]))

## 02. A validated shopping cart

An object owns its state and enforces positive quantities and nonnegative prices. Explicit methods are the public interface.

**Acceptance criteria:** Independent instances, validated item operations.

**Hint:** Do not put item storage at class level.

In [ ]:
class Cart:
    def __init__(self):
        self._items = {}
    def add(self, sku, unit_price, quantity=1):
        if not sku or unit_price < 0 or not isinstance(quantity, int) or quantity <= 0:
            raise ValueError("invalid item")
        if sku in self._items and self._items[sku][0] != unit_price:
            raise ValueError("price mismatch for existing SKU")
        old = self._items.get(sku, (unit_price, 0))[1]
        self._items[sku] = (unit_price, old + quantity)
    def total(self):
        return sum(price * quantity for price, quantity in self._items.values())

cart = Cart()
cart.add("P1", 5.0, 2)
cart.add("P1", 5.0)
assert cart.total() == 15
print(cart.total())

## 03. Composed report writer

Pass a formatter dependency into a report service instead of extending the formatter as a base class.

**Acceptance criteria:** Replace formatter without changing report service.

**Hint:** Real CSV with escaped commas should use the csv module.

In [ ]:
class CsvFormatter:
    def format(self, rows):
        return "\n".join(",".join(str(value) for value in row) for row in rows)

class ReportService:
    def __init__(self, formatter):
        self.formatter = formatter
    def generate(self, rows):
        return self.formatter.format(rows)

report = ReportService(CsvFormatter())
assert report.generate([("Ada", 91), ("Ben", 82)]) == "Ada,91\nBen,82"
print(report.generate([("A", 1)]))

## 04. Dataclass with private collection

Use a factory for per-instance mutable state and post-init validation.

**Acceptance criteria:** No shared list between instances.

**Hint:** A mutable default list can leak state.

In [ ]:
from dataclasses import dataclass, field
@dataclass
class Order:
    id: str
    quantities: list[int] = field(default_factory=list)
    def __post_init__(self):
        if not self.id:
            raise ValueError("id required")
    @property
    def units(self):
        return sum(self.quantities)

a, b = Order("A"), Order("B")
a.quantities.append(3)
assert a.units == 3
assert b.units == 0

## 05. Safe aggregate prices

Return a new mapping rather than altering the input collection.

**Acceptance criteria:** Repeated SKUs aggregated without mutation.

**Hint:** Dictionary accumulator.

In [ ]:
def total_by_sku(lines):
    totals = {}
    for sku, quantity, price in lines:
        if not sku or quantity <= 0 or price < 0:
            raise ValueError("invalid line")
        totals[sku] = totals.get(sku, 0) + quantity * price
    return totals

assert total_by_sku([("A", 2, 5), ("A", 1, 5), ("B", 1, 3)]) == {"A": 15, "B": 3}

## 06. Property with validation

Make humidity a validated attribute with a controlled setter.

**Acceptance criteria:** Invalid assignment rejected.

**Hint:** Do not store a second unvalidated public field.

In [ ]:
class Sensor:
    def __init__(self, humidity):
        self.humidity = humidity
    @property
    def humidity(self):
        return self._humidity
    @humidity.setter
    def humidity(self, value):
        if not 0 <= value <= 100:
            raise ValueError("humidity must be 0 to 100")
        self._humidity = float(value)

sensor = Sensor(45)
sensor.humidity = 61
assert sensor.humidity == 61
try:
    sensor.humidity = 150
except ValueError:
    print("Invalid humidity rejected")

## Reflection

- Which solution uses a reusable function rather than hidden notebook state?
- Have you tested normal, boundary, and invalid cases?
- What is the expected complexity or important design trade-off?
- Can you restart the kernel and run all cells cleanly?

**Reference notebooks:** See `../solutions/` for a selection of worked examples. The solutions deliberately do not cover every challenge; explain and test the remaining problems independently.